In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

Matplotlib is building the font cache; this may take a moment.


In [ ]:
ds = pd.read_csv('data/baseline/dataset.csv', index_col = 0)
ds

,ticker,year,Выручка,Операционная прибыль,EBITDA,Чистая прибыль,Опер.денежный поток,CAPEX,FCF,"Див доход, ао",...,P/BV,EV/EBITDA,Долг/EBITDA,R&D/CAPEX,CAPEX/Выручка,total_price,total_price_target,revenue,total_price_momentum,momentum
0,ENPG,2022,1134.0,137.50,213.80,74.20,39.20,117.300,-39.10,0.0,...,0.67,4.45,3.33,0.0,10.0,374.000,440.200,0.162973,885.000,-0.861332
1,ENPG,2023,1249.0,87.80,183.90,50.80,232.00,123.400,58.00,0.0,...,0.65,5.78,4.25,0.0,10.0,440.200,347.300,-0.237040,374.000,0.162973
2,ENPG,2024,1335.0,137.20,266.70,90.80,151.10,171.100,-91.40,0.0,...,0.43,3.86,3.03,0.0,13.0,347.300,454.400,0.268789,440.200,-0.237040
3,HEAD,2022,18.1,6.92,9.16,6.01,7.67,0.450,7.00,0.0,...,-10.70,6.50,-0.19,0.0,2.0,1211.000,2951.000,0.890698,3899.408,-1.169378
4,HEAD,2023,29.4,15.80,17.40,12.50,16.40,0.517,16.40,0.0,...,293.00,7.48,-1.12,0.0,2.0,2951.000,4604.010,0.444784,1211.000,0.890698
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
551,YKEN,2022,25.3,12.10,5.30,8.00,0.67,5.950,-8.51,0.0,...,-3.27,6.64,6.18,0.0,23.0,0.264,0.609,0.835869,0.305,-0.144363
552,YKEN,2023,32.2,7.20,5.90,3.00,5.12,10.300,-9.47,0.0,...,1.00,7.51,6.33,0.0,32.0,0.609,0.480,-0.238032,0.264,0.835869
553,YDEX,2022,521.7,13.20,64.10,10.80,41.70,50.500,-10.90,0.0,...,4.66,9.72,-0.50,0.0,10.0,1814.000,2542.000,0.337417,4502.000,-0.908987
554,YDEX,2023,798.1,63.90,120.80,52.10,115.60,84.200,-37.70,0.0,...,7.35,7.97,0.37,0.0,11.0,2542.000,4137.120,0.487049,1814.000,0.337417


In [52]:
train = ds[ds["year"] <= 2023]
test = ds[ds["year"] >= 2024]
train.shape, test.shape

((366, 46), (78, 46))

In [53]:
train

,ticker,year,Выручка,Операционная прибыль,EBITDA,Чистая прибыль,Опер.денежный поток,CAPEX,FCF,"Див доход, ао",...,P/BV,EV/EBITDA,Долг/EBITDA,R&D/CAPEX,CAPEX/Выручка,total_price,total_price_target,revenue,total_price_momentum,momentum
0,ENPG,2022,1134.0,137.50,213.80,74.20,39.20,117.300,-39.10,0.0,...,0.67,4.45,3.33,0.0,10.0,374.000,440.2000,0.162973,885.000,-0.861332
1,ENPG,2023,1249.0,87.80,183.90,50.80,232.00,123.400,58.00,0.0,...,0.65,5.78,4.25,0.0,10.0,440.200,347.3000,-0.237040,374.000,0.162973
3,HEAD,2022,18.1,6.92,9.16,6.01,7.67,0.450,7.00,0.0,...,-10.70,6.50,-0.19,0.0,2.0,1211.000,2951.0000,0.890698,3899.408,-1.169378
4,HEAD,2023,29.4,15.80,17.40,12.50,16.40,0.517,16.40,0.0,...,293.00,7.48,-1.12,0.0,2.0,2951.000,4604.0100,0.444784,1211.000,0.890698
6,HNFG,2022,12.4,3.03,3.12,1.80,1.68,1.950,-1.19,0.0,...,23.40,9.03,1.24,0.0,16.0,675.000,552.6675,-0.199956,675.000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
550,YKEN,2021,22.6,-9.43,4.19,-10.20,4.43,461.000,-3.00,0.0,...,-0.32,6.47,5.79,0.0,2037.0,0.305,0.2640,-0.144363,0.380,-0.219859
551,YKEN,2022,25.3,12.10,5.30,8.00,0.67,5.950,-8.51,0.0,...,-3.27,6.64,6.18,0.0,23.0,0.264,0.6090,0.835869,0.305,-0.144363
552,YKEN,2023,32.2,7.20,5.90,3.00,5.12,10.300,-9.47,0.0,...,1.00,7.51,6.33,0.0,32.0,0.609,0.4800,-0.238032,0.264,0.835869
553,YDEX,2022,521.7,13.20,64.10,10.80,41.70,50.500,-10.90,0.0,...,4.66,9.72,-0.50,0.0,10.0,1814.000,2542.0000,0.337417,4502.000,-0.908987


In [54]:
X = train[[
        "momentum",
        "P/E",
        "ROE",
        "P/BV",
        "EV/EBITDA",
        "Долг/EBITDA",
        "R&D/CAPEX",
        "CAPEX/Выручка"
    ]]

(X.max() - X.min()) / X.mean()

momentum          40.049535
P/E              375.982712
ROE              228.208528
P/BV             387.072115
EV/EBITDA        137.594770
Долг/EBITDA      221.853489
R&D/CAPEX        152.617605
CAPEX/Выручка    125.575543
dtype: float64

In [55]:
from sklearn.preprocessing import QuantileTransformer

qt = QuantileTransformer(n_quantiles=100, output_distribution='normal', random_state=42)
X = qt.fit_transform(X)

In [76]:
# OLS с таргетом revenue

lin_reg = sm.OLS(
    train["revenue"],
    sm.add_constant(X)
).fit()
train["lin_reg_pred"] = lin_reg.predict()
lin_reg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                revenue   R-squared:                       0.108
Model:                            OLS   Adj. R-squared:                  0.088
Method:                 Least Squares   F-statistic:                     5.415
Date:                Sun, 05 Apr 2026   Prob (F-statistic):           1.89e-06
Time:                        18:27:22   Log-Likelihood:                -393.75
No. Observations:                 366   AIC:                             805.5
Df Residuals:                     357   BIC:                             840.6
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0031      0.119      0.026      0.979      -0.232       0.238
x1            -0.1658      0.036     -4.581      0.000      -0.237      -0.095
x2             0.0178      0.044      0.408      0.683      -0.068       0.104
x3            -0.0338      0.041     -0.814      0.416      -0.115       0.048
x4            -0.0243      0.046     -0.526      0.599      -0.115       0.067
x5            -0.0910      0.047     -1.921      0.056      -0.184       0.002
x6            -0.0268      0.042     -0.635      0.526      -0.110       0.056
x7            -0.0011      0.024     -0.048      0.961      -0.047       0.045
x8            -0.0799      0.023     -3.479      0.001      -0.125      -0.035
==============================================================================
Omnibus:                      343.649   Durbin-Watson:                   1.870
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            14586.278
Skew:                          -3.781   Prob(JB):                         0.00
Kurtosis:                      32.988   Cond. No.                         16.8
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [77]:
# OLS с таргетом revenue

log_reg = sm.Logit(
    (train.groupby('year')['revenue'].rank(pct=True) >= 0.5).astype(np.int32),
    sm.add_constant(X)
).fit()
train["log_reg_pred"] = log_reg.predict()
log_reg.summary()

Optimization terminated successfully.
         Current function value: 0.669389
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                revenue   No. Observations:                  366
Model:                          Logit   Df Residuals:                      357
Method:                           MLE   Df Model:                            8
Date:                Sun, 05 Apr 2026   Pseudo R-squ.:                 0.03409
Time:                        18:27:40   Log-Likelihood:                -245.00
converged:                       True   LL-Null:                       -253.64
Covariance Type:            nonrobust   LLR p-value:                   0.02720
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0999      0.343     -0.291      0.771      -0.773       0.573
x1            -0.0381      0.103     -0.370      0.711      -0.240       0.164
x2            -0.0581      0.126     -0.463      0.644      -0.304       0.188
x3            -0.0045      0.120     -0.037      0.970      -0.239       0.230
x4            -0.0332      0.134     -0.248      0.804      -0.296       0.230
x5            -0.1998      0.137     -1.457      0.145      -0.469       0.069
x6            -0.1241      0.121     -1.022      0.307      -0.362       0.114
x7            -0.0192      0.068     -0.283      0.777      -0.152       0.114
x8            -0.1776      0.070     -2.536      0.011      -0.315      -0.040
==============================================================================
"""

In [ ]:
# Метрики на train датасете

lin_reg_long = train.groupby('year')['lin_reg_pred'].rank(pct=True) >= 0.8
lin_reg_short = train.groupby('year')['lin_reg_pred'].rank(pct=True) <= 0.2

log_reg_long = train.groupby('year')['log_reg_pred'].rank(pct=True) >= 0.8
log_reg_short = train.groupby('year')['log_reg_pred'].rank(pct=True) <= 0.2

true_long = train.groupby('year')['revenue'].rank(pct=True) >= 0.8
true_short = train.groupby('year')['revenue'].rank(pct=True) <= 0.2

In [ ]:
from sklearn.metrics import confusion_matrix

print("Lin reg long confusion matrix")
cm = confusion_matrix(
    lin_reg_long.astype(np.int32),
    true_long.astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

print("Lin reg short confusion matrix")
cm = confusion_matrix(
    lin_reg_short.astype(np.int32),
    true_short.astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

print("Log reg long confusion matrix")
cm = confusion_matrix(
    log_reg_long.astype(np.int32),
    true_long.astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

print("Log reg long confusion matrix")
cm = confusion_matrix(
    log_reg_short.astype(np.int32),
    true_short.astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

Lin reg long confusion matrix
[[242  48]
 [ 48  28]]
Precision: 0.3684210526315789, Recall: 0.3684210526315789, F1: 0.3684210526315789


Lin reg short confusion matrix
[[244  49]
 [ 49  24]]
Precision: 0.3287671232876712, Recall: 0.3287671232876712, F1: 0.3287671232876712


Log reg long confusion matrix
[[245  45]
 [ 45  31]]
Precision: 0.40789473684210525, Recall: 0.40789473684210525, F1: 0.40789473684210525


Log reg long confusion matrix
[[240  53]
 [ 53  20]]
Precision: 0.273972602739726, Recall: 0.273972602739726, F1: 0.273972602739726




In [95]:
# Доходность на train датасете

train["revenue"] = np.exp(train["revenue"])

In [112]:
lin_reg_revenue = (
    (train[lin_reg_long == 1].groupby('year')["revenue"].mean() - 1) # long
    + (1 - train[lin_reg_short == 1].groupby('year')["revenue"].mean()) # short
)

log_reg_revenue = (
    (train[log_reg_long == 1].groupby('year')["revenue"].mean() - 1) # long
    + (1 - train[log_reg_short == 1].groupby('year')["revenue"].mean()) # short
)

oracle_revenue = (
    (train[true_long == 1].groupby('year')["revenue"].mean() - 1) # long
    + (1 - train[true_short == 1].groupby('year')["revenue"].mean()) # short
)

print(f"Lin reg revenue: {lin_reg_revenue}")
print()
print(f"Log reg revenue: {log_reg_revenue}")
print()
print(f"Oracle revenue: {oracle_revenue}")

Lin reg revenue: year
2021    0.243661
2022    0.722221
2023    0.168248
Name: revenue, dtype: float64

Log reg revenue: year
2021    0.253198
2022    1.140391
2023    0.282599
Name: revenue, dtype: float64

Oracle revenue: year
2021    0.725235
2022    2.753994
2023    0.864451
Name: revenue, dtype: float64


In [ ]:
# предсказание на тестовом датасете
X_test = test[[
        "momentum",
        "P/E",
        "ROE",
        "P/BV",
        "EV/EBITDA",
        "Долг/EBITDA",
        "R&D/CAPEX",
        "CAPEX/Выручка"
    ]]
X_test = qt.transform(X_test)

test["lin_reg_pred"] = lin_reg.predict(sm.add_constant(X_test))
test["log_reg_pred"] = log_reg.predict(sm.add_constant(X_test))

In [121]:
test_lin_reg_long = test.groupby('year')['lin_reg_pred'].rank(pct=True) >= 0.8
test_lin_reg_short = test.groupby('year')['lin_reg_pred'].rank(pct=True) <= 0.2

test_log_reg_long = test.groupby('year')['log_reg_pred'].rank(pct=True) >= 0.8
test_log_reg_short = test.groupby('year')['log_reg_pred'].rank(pct=True) <= 0.2

test_true_long = test.groupby('year')['revenue'].rank(pct=True) >= 0.8
test_true_short = test.groupby('year')['revenue'].rank(pct=True) <= 0.2

In [ ]:
# метрики на тестовом датасете

print("Lin reg long confusion matrix")
cm = confusion_matrix(
    test_lin_reg_long.astype(np.int32),
    test_true_long.astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

print("Lin reg short confusion matrix")
cm = confusion_matrix(
    test_lin_reg_short.astype(np.int32),
    test_true_short.astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

print("Log reg long confusion matrix")
cm = confusion_matrix(
    test_log_reg_long.astype(np.int32),
    test_true_long.astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

print("Log reg long confusion matrix")
cm = confusion_matrix(
    test_log_reg_short.astype(np.int32),
    test_true_short.astype(np.int32)
)
print(cm)
pr = cm[1][1] / (cm[1][1] + cm[1][0])
re = cm[1][1] / (cm[1][1] + cm[0][1])
print(f"Precision: {pr}, Recall: {re}, F1: {2 / (1 / pr + 1 / re)}")
print()
print()

Lin reg long confusion matrix
[[48 14]
 [14  2]]
Precision: 0.125, Recall: 0.125, F1: 0.125


Lin reg short confusion matrix
[[51 12]
 [12  3]]
Precision: 0.2, Recall: 0.2, F1: 0.2


Log reg long confusion matrix
[[51 11]
 [11  5]]
Precision: 0.3125, Recall: 0.3125, F1: 0.3125


Log reg long confusion matrix
[[53 10]
 [10  5]]
Precision: 0.3333333333333333, Recall: 0.3333333333333333, F1: 0.3333333333333333




In [123]:
test["revenue"] = np.exp(test["revenue"])

In [126]:
# доходность на тестовом датасете

test_lin_reg_revenue = (
    (test[test_lin_reg_long == 1].groupby('year')["revenue"].mean() - 1) # long
    + (1 - test[test_lin_reg_short == 1].groupby('year')["revenue"].mean()) # short
)

test_log_reg_revenue = (
    (test[test_log_reg_long == 1].groupby('year')["revenue"].mean() - 1) # long
    + (1 - test[test_log_reg_short == 1].groupby('year')["revenue"].mean()) # short
)

test_oracle_revenue = (
    (test[test_true_long == 1].groupby('year')["revenue"].mean() - 1) # long
    + (1 - test[test_true_short == 1].groupby('year')["revenue"].mean()) # short
)

print(f"Lin reg revenue: {test_lin_reg_revenue}")
print()
print(f"Log reg revenue: {test_log_reg_revenue}")
print()
print(f"Oracle revenue: {test_oracle_revenue}")

Lin reg revenue: year
2024    0.044303
Name: revenue, dtype: float64

Log reg revenue: year
2024    0.278743
Name: revenue, dtype: float64

Oracle revenue: year
2024    0.792439
Name: revenue, dtype: float64


In [ ]:
# добавляем предикты бейзлайнов в датасет для гибридных подходов

ds["lin_reg_pred"] = lin_reg.predict(sm.add_constant(
    qt.transform(
        ds[[
                "momentum",
                "P/E",
                "ROE",
                "P/BV",
                "EV/EBITDA",
                "Долг/EBITDA",
                "R&D/CAPEX",
                "CAPEX/Выручка"
        ]]
)
))
ds["log_reg_pred"] = log_reg.predict(sm.add_constant(
    qt.transform(
        ds[[
                "momentum",
                "P/E",
                "ROE",
                "P/BV",
                "EV/EBITDA",
                "Долг/EBITDA",
                "R&D/CAPEX",
                "CAPEX/Выручка"
        ]]
)
))
ds.to_csv("data/baseline/enriched_dataset.csv")